In [2]:
# ================================= TASK 1: Dataset Understanding =================================

print("=== TASK 1: Dataset Understanding ===\n")

import pandas as pd

df = pd.read_csv('/content/customer_support_text_classification.csv')

print(" Dataset loaded successfully!\n")
print("Number of records:", len(df))
print("\nColumns in dataset:", df.columns.tolist())

# Correct columns
text_col = 'customer_message'
label_col = 'sentiment_label'

print("\nTarget classes (sentiment_label):", sorted(df[label_col].unique()))

print("\nSample Text Records:")
print("Record 1:", df[text_col].iloc[0])
print("\nRecord 2:", df[text_col].iloc[1])

df['text_length'] = df[text_col].astype(str).apply(lambda x: len(x.split()))
print("\nAverage text length (in words):", round(df['text_length'].mean(), 2))

print("\nClass Distribution:")
print(df[label_col].value_counts())
print("\nClass Distribution (%):")
print(round(df[label_col].value_counts(normalize=True) * 100, 2))

=== TASK 1: Dataset Understanding ===

 Dataset loaded successfully!

Number of records: 1500

Columns in dataset: ['ticket_id', 'channel', 'customer_message', 'sentiment_label', 'word_count', 'urgent_flag']

Target classes (sentiment_label): ['negative', 'neutral', 'positive']

Sample Text Records:
Record 1: I need information about the payment process. My ticket number is 78732. Please respond as soon as possible.

Record 2: I need information about the payment process.

Average text length (in words): 12.72

Class Distribution:
sentiment_label
neutral     524
negative    497
positive    479
Name: count, dtype: int64

Class Distribution (%):
sentiment_label
neutral     34.93
negative    33.13
positive    31.93
Name: proportion, dtype: float64


Key Findings:

Number of records: 1500
Target labels/classes: There are three sentiment classes - 'negative', 'neutral', and 'positive'
Sample text records:
Record 1: I need information about the payment process. My ticket number is 78732. Please respond as soon as possible.
Record 2: I need information about the payment process.

Average text length: 12.72 words
Class distribution:
- neutral: 524 records (34.93%)
- negative: 497 records (33.13%)
- positive: 479 records (31.93%)

In [3]:
# ================================= TASK 2: Text Preprocessing =================================

print("=== TASK 2: Text Preprocessing ===\n")

import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

# Download NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

# Load dataset
df = pd.read_csv('/content/customer_support_text_classification.csv')
text_column = 'customer_message'

df['cleaned_text'] = df[text_column].astype(str)

# 1. Lowercasing
df['cleaned_text'] = df['cleaned_text'].str.lower()

# 2. Removing symbols/special characters
df['cleaned_text'] = df['cleaned_text'].apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]', '', x))

# 3. Tokenization + Remove stopwords
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words]
    return tokens

df['tokenized'] = df['cleaned_text'].apply(preprocess_text)

# 4. Padding / Truncating sequences (for sequence models)
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df['cleaned_text'])
sequences = tokenizer.texts_to_sequences(df['cleaned_text'])
padded_sequences = pad_sequences(sequences, maxlen=50, padding='post', truncating='post')

print("Original Sample:", df[text_column].iloc[0])
print("\nCleaned Text:", df['cleaned_text'].iloc[0])
print("Tokenized Text:", df['tokenized'].iloc[0])
print("Padded Sequence Sample (first 20):", padded_sequences[0][:20])


=== TASK 2: Text Preprocessing ===



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Original Sample: I need information about the payment process. My ticket number is 78732. Please respond as soon as possible.

Cleaned Text: i need information about the payment process my ticket number is 78732 please respond as soon as possible
Tokenized Text: ['need', 'information', 'payment', 'process', 'ticket', 'number', '78732', 'please', 'respond', 'soon', 'possible']
Padded Sequence Sample (first 20): [  4  29 136  39   1  90  33   3   6   7   2 184  10  12   8  13   8  14
   0   0]


In [4]:
# ================================= TASK 3: Text Vectorization =================================

print("=== TASK 3: Text Vectorization ===\n")

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load dataset
df = pd.read_csv('/content/customer_support_text_classification.csv')
text_column = 'customer_message'

# Use cleaned text from Task 2
df['cleaned_text'] = df[text_column].astype(str).str.lower()
df['cleaned_text'] = df['cleaned_text'].apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]', '', x))

print("Sample Text for Vectorization:")
print(df['cleaned_text'].iloc[0])
print("\n")

# 1. Bag of Words (BoW)
bow_vectorizer = CountVectorizer(max_features=500)
bow_matrix = bow_vectorizer.fit_transform(df['cleaned_text'])
print("Bag of Words - Shape:", bow_matrix.shape)
print("BoW Feature Names (first 10):", bow_vectorizer.get_feature_names_out()[:10])

# 2. TF-IDF
tfidf_vectorizer = TfidfVectorizer(max_features=500)
tfidf_matrix = tfidf_vectorizer.fit_transform(df['cleaned_text'])
print("\nTF-IDF - Shape:", tfidf_matrix.shape)

# 3. Tokenizer-based Sequences (for Deep Learning)
tokenizer = Tokenizer(num_words=1000)
tokenizer.fit_on_texts(df['cleaned_text'])
sequences = tokenizer.texts_to_sequences(df['cleaned_text'])
padded_sequences = pad_sequences(sequences, maxlen=50, padding='post')

print("\nTokenizer-based Sequences - Shape:", padded_sequences.shape)
print("Sample Padded Sequence:", padded_sequences[0][:15])

=== TASK 3: Text Vectorization ===

Sample Text for Vectorization:
i need information about the payment process my ticket number is 78732 please respond as soon as possible


Bag of Words - Shape: (1500, 500)
BoW Feature Names (first 10): ['11045' '11058' '11213' '11482' '11855' '12223' '12238' '12408' '12727'
 '12970']

TF-IDF - Shape: (1500, 500)

Tokenizer-based Sequences - Shape: (1500, 50)
Sample Padded Sequence: [  4  29 136  39   1  90  33   3   6   7   2 184  10  12   8]


**Explanation**

Text must be converted into vectors before being used by a machine learning or deep learning model because computers can only understand and process numerical data. Machine learning algorithms cannot work directly with raw text. Converting text into numbers (vectors) allows the model to find patterns, relationships, and meanings in the text data, which is essential for tasks like sentiment classification.

In [5]:
# ================================= TASK 4: Baseline Model =================================

print("=== TASK 4: Baseline Model ===\n")

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Load dataset
df = pd.read_csv('/content/customer_support_text_classification.csv')

# Use cleaned text
df['cleaned_text'] = df['customer_message'].astype(str).str.lower()
df['cleaned_text'] = df['cleaned_text'].apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]', '', x))

X = df['cleaned_text']
y = df['sentiment_label']

# Vectorization
tfidf = TfidfVectorizer(max_features=1000)
X_vectorized = tfidf.fit_transform(X)

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_vectorized, y, test_size=0.2, random_state=42)

# Build Baseline Model - Logistic Regression
model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)

# Predictions
y_pred = model.predict(X_test)

# Evaluation
accuracy = accuracy_score(y_test, y_pred)
print("Baseline Model Accuracy:", round(accuracy * 100, 2), "%\n")
print("Classification Report:\n")
print(classification_report(y_test, y_pred))

=== TASK 4: Baseline Model ===

Baseline Model Accuracy: 100.0 %

Classification Report:

              precision    recall  f1-score   support

    negative       1.00      1.00      1.00       109
     neutral       1.00      1.00      1.00       104
    positive       1.00      1.00      1.00        87

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300



I built a simple baseline model using Logistic Regression with TF-IDF vectorization.
Steps Performed:

- Converted the cleaned text into TF-IDF vectors
Split the data into training and testing sets (80-20 split)
- Trained a Logistic Regression model
Evaluated the model using Accuracy, Precision, Recall, and F1-score

- The baseline model achieved perfect accuracy of 100% on the test set. This shows that the TF-IDF features were highly effective in distinguishing between the sentiment classes for this dataset.

In [6]:
# ================================= TASK 5: Sequence Model =================================

print("=== TASK 5: Sequence Model (LSTM) ===\n")

import pandas as pd
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Load data
df = pd.read_csv('/content/customer_support_text_classification.csv')

# Preprocessing
texts = df['customer_message'].astype(str).str.lower()
texts = texts.apply(lambda x: re.sub(r'[^a-zA-Z0-9\s]', '', x))

# Tokenization & Padding
tokenizer = Tokenizer(num_words=2000)
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
X = pad_sequences(sequences, maxlen=50, padding='post')

# Encode labels
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['sentiment_label'])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Build LSTM Model
model = Sequential()
model.add(Embedding(input_dim=2000, output_dim=100, input_length=50))
model.add(LSTM(64))
model.add(Dense(3, activation='softmax'))

model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

print("\n LSTM Sequence Model Architecture Built Successfully!")

=== TASK 5: Sequence Model (LSTM) ===



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)


 LSTM Sequence Model Architecture Built Successfully!


For this task, I built a simple LSTM-based sequence model for sentiment classification of customer messages.
I first converted the cleaned text into sequences using a Tokenizer and applied padding to make all sequences of length 50. The labels (negative, neutral, positive) were encoded into numbers.
The model architecture consists of:

An Embedding layer that turns words into 100-dimensional vectors
An LSTM layer with 64 units to understand the sequence and context of words
A Dense output layer with 3 neurons and softmax activation to predict one of the three sentiment classes

I used sparse categorical crossentropy as the loss function and accuracy as the main evaluation metric.
This sequence model is especially useful for text data because it can understand the order of words and the context, which traditional methods like Bag of Words cannot capture properly. This helps in better understanding the meaning behind customer messages.

Task 6: Attention and Transformer Reflection

Q1. Why do RNNs struggle with long-term dependencies?

RNNs often struggle with long-term dependencies because as the sequence becomes longer, they tend to forget the information from the earlier parts of the text by the time they reach the end. This happens mainly due to the vanishing gradient problem during training.

Q2. How do LSTMs help with memory?

LSTMs help with memory by using special gates (forget gate, input gate, and output gate). These gates allow the model to decide what information to keep, what to update, and what to throw away. This mechanism enables LSTMs to remember important context even over longer sequences.

Q3. What does attention solve in sequence-to-sequence tasks?

Attention solves the problem of relying only on the final hidden state. It allows the model to focus on the most relevant parts of the input sequence while making predictions. This helps the model better understand which words are more important for the current task.

Q4. Why are Transformers important in modern NLP and Generative AI?

Transformers are important because they completely remove recurrence and use self-attention mechanism. This allows them to process all words in parallel, capture relationships between words no matter how far apart they are, and train much faster. Because of this, they have become the foundation for modern NLP models like BERT and GPT used in Generative AI.
